In [ ]:
!pip install -q langchain langchain-chroma langchain-community langchain-openai langchain-text-splitters wikipedia

In [ ]:
from google.colab import userdata
from itertools import batched
from langchain.agents import create_agent
from langchain_chroma import Chroma
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_community.retrievers import WikipediaRetriever
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.tools import create_retriever_tool
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pydantic import SecretStr
from typing import List


api_key = SecretStr(userdata.get('OPENAI_API_KEY'))


def print_conversation(conversation: List[AIMessage]):
    for message in conversation:
        message.pretty_print()

You can download books in .txt format from the [Gutenberg Project](https://www.gutenberg.org/browse/scores/top).

In [ ]:
# NOTE: By default, if `loader_cls` is not provided, the `UnstructuredFileLoader` will be used (which requires installing the `unstructured` package in addition).
directory_loader = DirectoryLoader("/content/books", glob="*.txt", loader_cls=TextLoader, loader_kwargs={"encoding": "utf-8"})
books = directory_loader.load()

In [ ]:
len(books)

In [ ]:
recursive_text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
split_book_chunks = recursive_text_splitter.split_documents(books)

In [ ]:
len(split_book_chunks)

In [ ]:
vector_store = Chroma(collection_name="books", persist_directory="/content/chroma")

# NOTE: As there are too many chunks, add them in batches.
for index, batch in enumerate(batched(split_book_chunks, 256)):
    vector_store.add_documents(batch)
    print(f"Processed batch #{index + 1}")

In [ ]:
quote_retriever = vector_store.as_retriever()

In [ ]:
quote_retriever.invoke(input="A Day Without Laughter Is a Day Wasted")

In [ ]:
# NOTE: We will use the summary only, so there is no need to fetch the entire Wikipedia page.
wikipedia_retriever = WikipediaRetriever(tok_k_results=5, doc_content_chars_max=0)

In [ ]:
wikipedia_retriever.invoke(input="Anna Karenina")

In [ ]:
agent = create_agent(
    model=ChatOpenAI(model="gpt-5-nano", api_key=api_key),
    tools=[
        create_retriever_tool(
            quote_retriever,
            name="lookup_quote",
            description="Search the book database for semantically relevant quotes.",
            document_separator="\n=====\n",
            document_prompt=PromptTemplate(input_variables=["source", "page_content"], template="Source: {source}\n\n{page_content}")
        ),
        create_retriever_tool(
            wikipedia_retriever,
            name="lookup_info",
            description="""
                        Search for information about authors or books.
                        Always use this tool after finding a quote to provide context about the author or the book.
                        The query MUST be a short keyword like an author name or book title, NOT a full sentence.
                        """,
            document_separator="\n=====\n",
            document_prompt=PromptTemplate(input_variables=["title", "summary"], template="Article: {title}\n\n{summary}")
        )
    ],
    system_prompt="""
                  You are an expert literary assistant with deep knowledge of world literature, poetry, and prose.
                  Use the \"lookup_quote\" tool for every interaction with the user to find relevant quotes.
                  Then, extend your knowledge (for the ones you like the most) using the \"lookup_info\" tool.
                  """
)

In [ ]:
meaning_of_love = agent.invoke(
    input={
        "messages": [HumanMessage("Is love a moral force or simply a powerful emotion?")]
    }
)

In [ ]:
print_conversation(meaning_of_love["messages"])